# Performing Inference

In [ ]:
import jax
import jax.numpy as jnp
from jax.sharding import Mesh
from jax.sharding import NamedSharding

import flax

from rich import print
import numpy as np
import os

from torch_to_flax import torch_to_flax
from model_flax import shard_params, Qwen2Config, Qwen2ForCausalLM

# Set this environment variable before importing JAX.
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.9375"


SHARD_MODEL = True


def init_model(SHARD_MODEL:bool):
    """Initialize a model with parameters on CPU then shard them to GPU after checkpoint loading."""
    # Create config and model
    config = Qwen2Config()
    model = Qwen2ForCausalLM(config=config)

    rng = jax.random.PRNGKey(0)
    input_shape = (1, 32)

    # Force initialization on CPU to avoid duplicate GPU allocations
    with jax.default_device(jax.devices("cpu")[0]):
        try:
            params = model.init(rng, jnp.ones(input_shape, dtype=jnp.int4))
        except Exception as e:
            params = model.init(rng, jnp.ones(input_shape, dtype=jnp.int32))


    # Load the parameters from the file
    try:
        with open("flax_params.msgpack", "rb") as f:
            params = {
                "params": flax.serialization.from_bytes(params["params"], f.read())
            }
    except FileNotFoundError:
        print("File not found. Running conversion...")
        torch_to_flax()
        with open("flax_params.msgpack", "rb") as f:
            params = {
                "params": flax.serialization.from_bytes(params["params"], f.read())
            }

    # Shard the parameters
    if SHARD_MODEL:
        params = shard_params(params)
    else:
        params = jax.device_put(params)


    return model, params

model, params = init_model(SHARD_MODEL)
print("Sharded model initialized.")


: 

In [ ]:
from transformers import AutoTokenizer
model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)


prompt = "Write a Python function that checks if a number is prime.\n"
inputs = tokenizer(prompt, return_tensors="pt")
input_ids = jnp.array(inputs["input_ids"].numpy())
# Generate
print("Generating tokens...")
output = model.generate(
    params,
    input_ids,
    max_new_tokens=100,
    temperature=0.7,
    do_sample=True,
    prng_key=jax.random.PRNGKey(0),
)

# Decode using your tokenizer
decoded = tokenizer.decode(np.array(output[0]))
print("Decoded text:", decoded)